# FraudIA Claims — Entrenamiento de Modelos ML

**Notebook para Google Colab**

Entrena los modelos de detección de fraude y exporta los artefactos:
- `Isolation Forest` — detección de anomalías (sin etiquetas)
- `Random Forest` — clasificación supervisada (etiqueta simulada)
- `SHAP` — explicabilidad de variables

**Salidas:**
```
models/fraud_model.pkl
models/scaler.pkl
models/model_columns.json
models/metrics.json
models/shap_feature_importance.json
```

## 0. Configuración del entorno

In [ ]:
# Instalar dependencias adicionales en Colab
!pip install shap xgboost -q

In [ ]:
# Montar Google Drive para leer datos y guardar modelos
from google.colab import drive
drive.mount('/content/drive')

# Ajusta esta ruta a donde tengas el proyecto en Drive
PROJECT_ROOT = '/content/drive/MyDrive/hackiathon-aseguradora-del-sur'

import sys
sys.path.insert(0, PROJECT_ROOT)

print(f'Proyecto: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import warnings
from pathlib import Path

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score
)
import shap
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
MODELS_DIR = Path(PROJECT_ROOT) / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Librerías cargadas correctamente')

## 1. Carga y preparación de datos

In [ ]:
DATA_PATH = Path(PROJECT_ROOT) / 'data' / 'processed' / 'claims_with_documents.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape[0]} siniestros, {df.shape[1]} columnas')
df.head(3)

In [ ]:
# Variables de entrada del modelo
FEATURE_COLS = [
    # Datos del siniestro
    'monto_reclamado',
    'monto_estimado',
    'dias_desde_inicio_poliza',
    'dias_hasta_fin_poliza',
    'dias_ocurrencia_reporte',
    'historial_siniestros_asegurado',
    'similitud_narrativa',
    'ratio_monto_suma',
    'cantidad_documentos',
    # Asegurado
    'n_reclamos_12_meses',
    'n_reclamos_historico',
    'reclamos_rc_sin_tercero',
    'antiguedad_anios',
    # Proveedor
    'n_siniestros_proveedor',
    'promedio_monto_proveedor',
    # Señales documentales (de PDFs)
    'doc_factura_alterada',
    'doc_ruc_invalido',
    'doc_parte_tardio',
    'doc_sin_denuncia_previa',
    'doc_sin_testigos',
    'doc_robo',
    'doc_perdida_total',
    # Alertas ya calculadas
    'proveedor_lista_restrictiva',
    'alerta_borde_inicio',
    'alerta_borde_fin',
    'reporte_tardio',
    'narrativa_similar',
    'narrativa_clonada',
]

# Filtrar columnas que efectivamente existen
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f'Features disponibles: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

In [ ]:
# Preparar matriz de features
X = df[FEATURE_COLS].copy()

# Convertir booleanos a int
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

# Imputar nulos con mediana
X = X.fillna(X.median(numeric_only=True))

print(f'X shape: {X.shape}')
print(f'Nulos restantes: {X.isnull().sum().sum()}')

In [ ]:
# Etiqueta simulada para Random Forest
# Un siniestro es 'riesgo alto' si cumple al menos una condición crítica

def create_label(row):
    if row.get('doc_factura_alterada', False):          return 1
    if row.get('doc_ruc_invalido', False):              return 1
    if row.get('proveedor_lista_restrictiva', False):   return 1
    sim = row.get('similitud_narrativa', 0) or 0
    dias = row.get('dias_desde_inicio_poliza', 999) or 999
    if sim >= 0.85 and dias <= 30:                      return 1
    if row.get('narrativa_clonada', False):              return 1
    if row.get('doc_sin_denuncia_previa', False) and row.get('doc_robo', False): return 1
    return 0

y = df.apply(create_label, axis=1)
print(f'Distribución de etiquetas:')
print(y.value_counts())
print(f'Proporción positivos: {y.mean():.2%}')

## 2. Isolation Forest (detección de anomalías)

In [ ]:
# Escalar para Isolation Forest
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaler ajustado')

In [ ]:
# Entrenar Isolation Forest
# contamination: fracción esperada de anomalías (~15% del dataset)
isof = IsolationForest(
    n_estimators=200,
    contamination=0.15,
    random_state=42,
    n_jobs=-1,
)
isof.fit(X_scaled)

# Score de anomalía: negativo = más anómalo
# Convertir a escala 0-100: 0 = normal, 100 = muy anómalo
anomaly_scores_raw = isof.score_samples(X_scaled)
score_min, score_max = anomaly_scores_raw.min(), anomaly_scores_raw.max()
isof_scores = ((score_min - anomaly_scores_raw) / (score_min - score_max) * 100).clip(0, 100)

df['score_isolation_forest'] = isof_scores.round(1)
print(f'Isolation Forest — score medio: {isof_scores.mean():.1f}')
print(f'Detectados como anómalos (>50): {(isof_scores > 50).sum()}')

## 3. Random Forest (clasificación supervisada)

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Random Forest con balanceo de clases
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

# Predicciones
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('Random Forest entrenado')
print(classification_report(y_test, y_pred, target_names=['Legítimo', 'Sospechoso']))

In [ ]:
# Métricas
metrics = {
    'precision':  round(float(precision_score(y_test, y_pred)),  3),
    'recall':     round(float(recall_score(y_test, y_pred)),     3),
    'f1':         round(float(f1_score(y_test, y_pred)),         3),
    'auc_roc':    round(float(roc_auc_score(y_test, y_proba)),   3),
    'cv_f1_mean': round(float(cross_val_score(rf, X, y, cv=5, scoring='f1').mean()), 3),
    'n_train':    int(len(X_train)),
    'n_test':     int(len(X_test)),
    'n_features': int(len(FEATURE_COLS)),
    'feature_cols': FEATURE_COLS,
}
print(json.dumps(metrics, indent=2))

In [ ]:
# Score RF para todos los siniestros (probabilidad clase 1)
rf_scores_all = rf.predict_proba(X)[:, 1] * 100
df['score_random_forest'] = rf_scores_all.round(1)
print(f'RF score medio: {rf_scores_all.mean():.1f}')
print(f'RF score máx:   {rf_scores_all.max():.1f}')

## 4. Explicabilidad con SHAP

In [ ]:
# SHAP TreeExplainer sobre Random Forest
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# shap_values[1] = valores para clase positiva (sospechoso)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

# Feature importance media |SHAP|
shap_importance = dict(
    sorted(
        zip(FEATURE_COLS, np.abs(sv).mean(axis=0).tolist()),
        key=lambda x: x[1],
        reverse=True,
    )
)
shap_importance = {k: round(v, 6) for k, v in shap_importance.items()}
print('Top 10 features por importancia SHAP:')
for feat, val in list(shap_importance.items())[:10]:
    print(f'  {feat:40s}: {val:.4f}')

In [ ]:
# Gráfico SHAP summary
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False, max_display=15)
plt.title('Importancia SHAP — FraudIA Claims')
plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado')

## 5. Exportación de artefactos

In [ ]:
# Guardar modelo y scaler
joblib.dump(rf,     MODELS_DIR / 'fraud_model.pkl')
joblib.dump(isof,   MODELS_DIR / 'isolation_forest.pkl')
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')

# Columnas del modelo
(MODELS_DIR / 'model_columns.json').write_text(
    json.dumps(FEATURE_COLS, ensure_ascii=False, indent=2)
)

# Métricas
(MODELS_DIR / 'metrics.json').write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2)
)

# SHAP feature importance
(MODELS_DIR / 'shap_feature_importance.json').write_text(
    json.dumps(shap_importance, ensure_ascii=False, indent=2)
)

print('Artefactos exportados:')
for f in sorted(MODELS_DIR.glob('*')):
    size = f.stat().st_size / 1024
    print(f'  {f.name}: {size:.1f} KB')

In [ ]:
# Guardar CSV con scores del modelo para todos los siniestros
scores_output = Path(PROJECT_ROOT) / 'data' / 'processed' / 'model_scores.csv'
df[['id_siniestro', 'score_isolation_forest', 'score_random_forest']].to_csv(
    scores_output, index=False, encoding='utf-8-sig'
)
print(f'Scores del modelo exportados: {scores_output}')

## 6. Verificación final

In [ ]:
# Verificar que los artefactos son cargables
rf_loaded    = joblib.load(MODELS_DIR / 'fraud_model.pkl')
scaler_loaded = joblib.load(MODELS_DIR / 'scaler.pkl')

# Prueba sobre SIN-0005 (caso de mayor riesgo)
sin0005 = X[df['id_siniestro'] == 'SIN-0005']
if len(sin0005) > 0:
    prob = rf_loaded.predict_proba(sin0005)[0, 1]
    print(f'SIN-0005 probabilidad de riesgo: {prob:.2%}')

print()
print('=== Resumen de entrenamiento ===')
print(f'  Precision:  {metrics["precision"]}')
print(f'  Recall:     {metrics["recall"]}')
print(f'  F1:         {metrics["f1"]}')
print(f'  AUC-ROC:    {metrics["auc_roc"]}')
print(f'  CV F1 mean: {metrics["cv_f1_mean"]}')
print()
print('Artefactos listos para usar en src/models/predict_model.py')